# ISTVT — Dataset
Loads preprocessed face-crop sequences for training and validation.

Each sample = **T consecutive frames from one video** + a real/fake label.  
Folders expected (built by `preprocess.ipynb`):
```
data/real/video_xxx/0000.jpg ...
data/fake/video_xxx/0000.jpg ...
```


In [ ]:
import os, random
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

## FaceSequenceDataset

In [ ]:
class FaceSequenceDataset(Dataset):
    """
    Samples a contiguous sequence of seq_len frames from each video folder.
    label 0 = real, 1 = fake.
    Train/val split is by VIDEO (not by frame) to prevent data leakage.
    """
    def __init__(self, root_dir, seq_len=6, img_size=128,
                 split="train", train_ratio=0.8, seed=42, augment=True):
        self.seq_len = seq_len
        self.split   = split
        self.samples = []

        for label, cls in [(0, "real"), (1, "fake")]:
            cls_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(cls_dir):
                continue
            for vid in sorted(os.listdir(cls_dir)):
                vid_dir = os.path.join(cls_dir, vid)
                if not os.path.isdir(vid_dir):
                    continue
                frames = sorted(
                    f for f in os.listdir(vid_dir)
                    if f.lower().endswith((".jpg", ".png"))
                )
                if len(frames) >= seq_len:
                    self.samples.append((vid_dir, frames, label))

        # deterministic split by video index
        rng = random.Random(seed)
        idx = list(range(len(self.samples)))
        rng.shuffle(idx)
        n_train = int(len(idx) * train_ratio)
        chosen  = idx[:n_train] if split == "train" else idx[n_train:]
        self.samples = [self.samples[i] for i in chosen]

        mean, std = [0.5]*3, [0.5]*3
        if augment and split == "train":
            self.transform = T.Compose([
                T.Resize((img_size, img_size)),
                T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
                T.ToTensor(),
                T.Normalize(mean, std),
            ])
        else:
            self.transform = T.Compose([
                T.Resize((img_size, img_size)),
                T.ToTensor(),
                T.Normalize(mean, std),
            ])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        vid_dir, frames, label = self.samples[idx]
        max_start = len(frames) - self.seq_len
        start = random.randint(0, max_start) if self.split == "train" else max_start // 2
        imgs = [
            self.transform(Image.open(os.path.join(vid_dir, frames[start + i])).convert("RGB"))
            for i in range(self.seq_len)
        ]
        return torch.stack(imgs), torch.tensor([label], dtype=torch.float32)

print("FaceSequenceDataset defined ✓")

## Create DataLoaders

In [ ]:
def make_loaders(root_dir, batch_size=4, seq_len=6, img_size=128, num_workers=2):
    train_ds = FaceSequenceDataset(root_dir, seq_len, img_size, split="train")
    val_ds   = FaceSequenceDataset(root_dir, seq_len, img_size, split="val", augment=False)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader

print("make_loaders defined ✓")

## Quick Check

In [ ]:
DATA_ROOT = "data"   # <-- set your path

train_loader, val_loader = make_loaders(DATA_ROOT, batch_size=4)
print(f"Train videos : {len(train_loader.dataset)}")
print(f"Val   videos : {len(val_loader.dataset)}")

seq, label = next(iter(train_loader))
print(f"Batch shape  : {seq.shape}")    # (4, 6, 3, 128, 128)
print(f"Label shape  : {label.shape}")  # (4, 1)